<a href="https://colab.research.google.com/github/aromapex/deepml-final-public/blob/main/deepml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
### Complete Chronological Match Pipeline & Dataset Splitting
#This notebook contains the complete pipeline to:
#1. Load all Premier League season files (`00_01.csv` to `26_27.csv`).
#2. Clean blank footer rows and malformed rows (extra trailing fields in 2003/04 and 2004/05), keeping only the target columns.
#3. Compute continuous pre-match Elo ratings over the full history (no leakage), with promoted teams starting at the average rating of the relegated teams and K, h tuned on the validation seasons.
#4. Split by season: 2000/01 burn-in (excluded), train 2001/02–2021/22, validation 2022/23–2023/24, test 2024/25 onwards.

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. Columns to keep (B365 odds are used ONLY as a benchmark, never as model inputs)
target_columns = [
    'Div', 'Date', 'HomeTeam', 'AwayTeam',
    'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR',
    'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC',
    'HY', 'AY', 'HR', 'AR',
    'B365H', 'B365D', 'B365A'
]

def season_files():
    """Yield (filename, path) for 00_01.csv .. 26_27.csv that exist."""
    for y in range(27):
        filename = f"{y:02d}_{y + 1:02d}.csv"
        path = f"/content/{filename}"
        if os.path.exists(path):
            yield filename, path

def load_season(path):
    """Read one season file; rows with extra trailing fields are truncated, not dropped."""
    with open(path, encoding='latin1') as f:
        n_cols = len(f.readline().rstrip('\r\n').split(','))
    return pd.read_csv(path, encoding='latin1', engine='python',
                       on_bad_lines=lambda row: row[:n_cols])

def parse_dates(s):
    """dd/mm/yy in older files, dd/mm/yyyy in newer."""
    d = pd.to_datetime(s, format='%d/%m/%y', errors='coerce')
    mask = d.isna()
    d[mask] = pd.to_datetime(s[mask], format='%d/%m/%Y', errors='coerce')
    return d

# 2. Load and combine
all_dfs = []
for filename, path in season_files():
    df = load_season(path)
    all_dfs.append(df[[c for c in target_columns if c in df.columns]].copy())

combined_raw_df = pd.concat(all_dfs, ignore_index=True)

# 3. Parse dates
combined_raw_df['Date_Parsed'] = parse_dates(combined_raw_df['Date'])

# 4. Drop blank/incomplete rows, then sort chronologically
n_before = len(combined_raw_df)
combined_raw_df = combined_raw_df.dropna(subset=['Date_Parsed', 'HomeTeam', 'AwayTeam', 'FTR'])
print(f"Dropped {n_before - len(combined_raw_df)} incomplete rows")
combined_raw_df = combined_raw_df.sort_values('Date_Parsed', kind='stable').reset_index(drop=True)

# Odds as numbers (some old files store them as text); missing odds stay NaN
for c in ['B365H', 'B365D', 'B365A']:
    combined_raw_df[c] = pd.to_numeric(combined_raw_df[c], errors='coerce')

# 5. Season column (July–June = one season)
def get_season(dt):
    y = dt.year if dt.month >= 7 else dt.year - 1
    return f"{y}/{str(y + 1)[-2:]}"

combined_raw_df['Season'] = combined_raw_df['Date_Parsed'].apply(get_season)

# Sanity check: results must be H/D/A only
bad = ~combined_raw_df['FTR'].isin(['H', 'D', 'A'])
assert not bad.any(), combined_raw_df[bad].head()

# 6. Elo with promoted teams starting at the average rating of the relegated teams
RESULT_TO_SCORE = {'H': 1.0, 'D': 0.5, 'A': 0.0}

def run_elo(df, K=20, h=80, init=1500.0):
    ratings, prev_teams = {}, set()
    home_elo = np.zeros(len(df))
    away_elo = np.zeros(len(df))

    for season, sdf in df.groupby('Season', sort=True):
        teams = set(sdf['HomeTeam']) | set(sdf['AwayTeam'])
        if not prev_teams:                                   # first season
            for t in teams:
                ratings[t] = init
        else:
            promoted = teams - prev_teams
            relegated = prev_teams - teams
            start = np.mean([ratings[t] for t in relegated]) if relegated else init
            for t in promoted:
                ratings[t] = start

        for i, home, away, res in zip(sdf.index, sdf['HomeTeam'], sdf['AwayTeam'], sdf['FTR']):
            r_h, r_a = ratings[home], ratings[away]
            home_elo[i], away_elo[i] = r_h, r_a              # pre-match ratings → no leakage
            s_h = RESULT_TO_SCORE[res]
            e_h = 1.0 / (1.0 + 10 ** ((r_a - r_h - h) / 400.0))
            ratings[home] = r_h + K * (s_h - e_h)
            ratings[away] = r_a - K * (s_h - e_h)
        prev_teams = teams

    return home_elo, away_elo

# 7. Tune K and h on the validation seasons only (Brier score of the home expectation)
VAL_SEASONS = ['2022/23', '2023/24']

def elo_score(df, K, h):
    he, ae = run_elo(df, K, h)
    e_h = 1.0 / (1.0 + 10 ** ((ae - he - h) / 400.0))
    s_h = df['FTR'].map(RESULT_TO_SCORE).to_numpy()
    val = df['Season'].isin(VAL_SEASONS).to_numpy()
    return np.mean((s_h[val] - e_h[val]) ** 2)

grid = [(K, h) for K in [10, 15, 20, 25, 30, 40] for h in [40, 60, 80, 100]]
best_K, best_h = min(grid, key=lambda p: elo_score(combined_raw_df, *p))
print(f"Best K = {best_K}, h = {best_h}")

# 8. Final Elo with tuned parameters
combined_raw_df['Home_Elo'], combined_raw_df['Away_Elo'] = run_elo(combined_raw_df, best_K, best_h)

# 9. Save master file
combined_raw_df.drop(columns=['Date_Parsed']).to_csv('/content/combined_data.csv', index=False)
print(f"Saved {len(combined_raw_df)} matches to /content/combined_data.csv")

Dropped 1 incomplete rows
Best K = 30, h = 60
Saved 9940 matches to /content/combined_data.csv


In [ ]:
# Split by season (2000/01 excluded from training as Elo burn-in)
all_seasons = sorted(combined_raw_df['Season'].unique())
print(f"All seasons found: {all_seasons}\n")

train_seasons = all_seasons[1:all_seasons.index('2022/23')]   # 2001/02–2021/22
val_seasons   = ['2022/23', '2023/24']
test_seasons  = [s for s in all_seasons if s >= '2024/25']

print(f"Train seasons: {train_seasons}")
print(f"Val seasons:   {val_seasons}")
print(f"Test seasons:  {test_seasons}\n")

def subset(seasons):
    return combined_raw_df[combined_raw_df['Season'].isin(seasons)].drop(columns=['Date_Parsed']).copy()

train_df_final = subset(train_seasons)
val_df_final   = subset(val_seasons)
test_df_final  = subset(test_seasons)

train_df_final.to_csv('/content/train_data.csv', index=False)
val_df_final.to_csv('/content/val_data.csv', index=False)
test_df_final.to_csv('/content/test_data.csv', index=False)

print(f"Train: {train_df_final.shape}, Val: {val_df_final.shape}, Test: {test_df_final.shape}")

All seasons found: ['2000/01', '2001/02', '2002/03', '2003/04', '2004/05', '2005/06', '2006/07', '2007/08', '2008/09', '2009/10', '2010/11', '2011/12', '2012/13', '2013/14', '2014/15', '2015/16', '2016/17', '2017/18', '2018/19', '2019/20', '2020/21', '2021/22', '2022/23', '2023/24', '2024/25', '2025/26', '2026/27']

Train seasons: ['2001/02', '2002/03', '2003/04', '2004/05', '2005/06', '2006/07', '2007/08', '2008/09', '2009/10', '2010/11', '2011/12', '2012/13', '2013/14', '2014/15', '2015/16', '2016/17', '2017/18', '2018/19', '2019/20', '2020/21', '2021/22']
Val seasons:   ['2022/23', '2023/24']
Test seasons:  ['2024/25', '2025/26', '2026/27']

Train: (7980, 28), Val: (760, 28), Test: (820, 28)


In [ ]:
### Sanity Checks and Verifications
## Below is the code verifying that the original dataset files were cleaned properly and containing no duplicates.

In [ ]:
# Verification: row counts and date ranges per season file
file_reports = []
for filename, path in season_files():
    df = load_season(path).dropna(subset=['Date']).copy()
    dates = parse_dates(df['Date'])
    file_reports.append({
        'Filename': filename,
        'Cleaned Rows': len(df),
        'Min Date': dates.min().strftime('%Y-%m-%d'),
        'Max Date': dates.max().strftime('%Y-%m-%d'),
    })

report_df = pd.DataFrame(file_reports)
display(report_df)
print(combined_raw_df[combined_raw_df.Season.isin(['2003/04', '2004/05'])].FTR.value_counts())

,Filename,Cleaned Rows,Min Date,Max Date
0,00_01.csv,380,2000-08-19,2001-05-19
1,01_02.csv,380,2001-08-18,2002-05-11
2,02_03.csv,380,2002-08-17,2003-05-11
3,03_04.csv,380,2003-08-16,2004-05-15
4,04_05.csv,380,2004-08-14,2005-05-15
5,05_06.csv,380,2005-08-13,2006-05-07
6,06_07.csv,380,2006-08-19,2007-05-13
7,07_08.csv,380,2007-08-11,2008-05-11
8,08_09.csv,380,2008-08-16,2009-05-24
9,09_10.csv,380,2009-08-15,2010-05-09


FTR
H    340
D    218
A    202
Name: count, dtype: int64


In [ ]:
# Verification: Confirm there are exactly 0 duplicate match mappings in the master raw DataFrame
duplicates = combined_raw_df.duplicated(subset=["Date", "HomeTeam", "AwayTeam"]).sum()
print(f"Total duplicates found across all combined files: {duplicates}")

Total duplicates found across all combined files: 0


In [ ]:
### Cleanup Summary
## We have organized the code to keep only the absolute necessary cells. Outdated duplicate code blocks have been removed and replaced with a sequential workflow.

In [ ]:
# Quick Preview of Training Data with Elo Ratings
display(train_df_final[['Season', 'Date', 'HomeTeam', 'AwayTeam', 'FTR', 'Home_Elo', 'Away_Elo']].head(10))

,Season,Date,HomeTeam,AwayTeam,FTR,Home_Elo,Away_Elo
380,2001/02,18/08/01,Charlton,Everton,A,1502.823997,1441.525223
381,2001/02,18/08/01,Derby,Blackburn,H,1463.138846,1397.440835
382,2001/02,18/08/01,Leeds,Southampton,H,1616.710423,1515.907201
383,2001/02,18/08/01,Leicester,Bolton,A,1419.141338,1397.440835
384,2001/02,18/08/01,Liverpool,West Ham,H,1605.840487,1428.804683
385,2001/02,18/08/01,Middlesbrough,Arsenal,A,1485.561423,1584.824383
386,2001/02,18/08/01,Sunderland,Ipswich,H,1519.064157,1566.334833
387,2001/02,18/08/01,Tottenham,Aston Villa,D,1480.764617,1516.266303
388,2001/02,19/08/01,Chelsea,Newcastle,D,1570.018943,1485.864631
389,2001/02,19/08/01,Man United,Fulham,H,1605.086007,1397.440835


In [ ]:
# Quick Preview of Validation Data with Elo Ratings
display(val_df_final[['Season', 'Date', 'HomeTeam', 'AwayTeam', 'FTR', 'Home_Elo', 'Away_Elo']].head(10))

,Season,Date,HomeTeam,AwayTeam,FTR,Home_Elo,Away_Elo
8360,2022/23,05/08/2022,Crystal Palace,Arsenal,A,1501.422156,1607.509855
8361,2022/23,06/08/2022,Fulham,Liverpool,D,1334.323418,1785.964973
8362,2022/23,06/08/2022,Bournemouth,Aston Villa,H,1334.323418,1471.176706
8363,2022/23,06/08/2022,Leeds,Wolves,H,1466.788825,1468.650589
8364,2022/23,06/08/2022,Newcastle,Nott'm Forest,H,1546.387454,1334.323418
8365,2022/23,06/08/2022,Tottenham,Southampton,H,1632.098820,1421.270916
8366,2022/23,06/08/2022,Everton,Chelsea,A,1448.187418,1630.049845
8367,2022/23,07/08/2022,Leicester,Brentford,D,1528.753744,1455.907449
8368,2022/23,07/08/2022,Man United,Brighton,A,1557.937033,1522.030245
8369,2022/23,07/08/2022,West Ham,Man City,A,1520.629919,1790.917682


In [ ]:
# Quick Preview of Testing Data with Elo Ratings
display(test_df_final[['Season', 'Date', 'HomeTeam', 'AwayTeam', 'FTR', 'Home_Elo', 'Away_Elo']].head(10))

,Season,Date,HomeTeam,AwayTeam,FTR,Home_Elo,Away_Elo
9120,2024/25,16/08/2024,Man United,Fulham,H,1579.696755,1477.783251
9121,2024/25,17/08/2024,Ipswich,Liverpool,A,1315.539544,1698.741316
9122,2024/25,17/08/2024,Arsenal,Wolves,H,1754.672847,1443.057189
9123,2024/25,17/08/2024,Everton,Brighton,A,1490.840212,1485.696263
9124,2024/25,17/08/2024,Newcastle,Southampton,H,1572.888704,1315.539544
9125,2024/25,17/08/2024,Nott'm Forest,Bournemouth,D,1411.342624,1463.780767
9126,2024/25,17/08/2024,West Ham,Aston Villa,A,1486.199507,1588.881242
9127,2024/25,18/08/2024,Brentford,Crystal Palace,H,1459.976349,1528.163142
9128,2024/25,18/08/2024,Chelsea,Man City,A,1598.958538,1808.740086
9129,2024/25,19/08/2024,Leicester,Tottenham,D,1315.539544,1562.616459


In [ ]:
import numpy as np, pandas as pd, torch, copy

df = combined_raw_df.drop(columns=['Date_Parsed']).copy()

# Stats per match, from each team's perspective: (home column, away column)
STATS = {'goals': ('FTHG', 'FTAG'), 'ht_goals': ('HTHG', 'HTAG'), 'shots': ('HS', 'AS'),
         'sot': ('HST', 'AST'), 'corners': ('HC', 'AC'), 'fouls': ('HF', 'AF'),
         'yellow': ('HY', 'AY'), 'red': ('HR', 'AR')}
STAT_COLS = [f'{s}_{k}' for s in STATS for k in ('for', 'against')]
FEAT_COLS = STAT_COLS + ['is_home', 'opp_elo']

def team_view(side):
    own = 0 if side == 'H' else 1
    v = pd.DataFrame({'match': df.index, 'team': df['HomeTeam' if side == 'H' else 'AwayTeam']})
    for s, cols in STATS.items():
        v[f'{s}_for'] = df[cols[own]].values
        v[f'{s}_against'] = df[cols[1 - own]].values
    v['is_home'] = 1.0 if side == 'H' else 0.0
    # Opponent's pre-match Elo in that past match (strength of schedule, no leakage)
    v['opp_elo'] = ((df['Away_Elo' if side == 'H' else 'Home_Elo'] - 1500) / 100).values
    return v

long = pd.concat([team_view('H'), team_view('A')]).sort_values('match', kind='stable')
long[STAT_COLS] = long[STAT_COLS].fillna(0).astype(float)

# Splits come from cell 2 (2000/01 = Elo burn-in, not used)
SPLITS = {'train': train_seasons, 'val': val_seasons, 'test': test_seasons}

# Normalise stats with TRAIN statistics only
train_rows = long['match'].map(df['Season']).isin(SPLITS['train'])
mu = long.loc[train_rows, STAT_COLS].mean()
sd = long.loc[train_rows, STAT_COLS].std()
long[STAT_COLS] = (long[STAT_COLS] - mu) / sd

# Position of each match in each team's own history
long['pos'] = long.groupby('team').cumcount()
team_hist = {t: g[FEAT_COLS].to_numpy(np.float32) for t, g in long.groupby('team')}
home_pos = long[long.is_home == 1].set_index('match')['pos'].reindex(df.index).to_numpy()
away_pos = long[long.is_home == 0].set_index('match')['pos'].reindex(df.index).to_numpy()

def build_windows(teams, positions, W):
    """[N, W, F+1]: last W matches strictly BEFORE each match, left-padded, + valid flag."""
    X = np.zeros((len(df), W, len(FEAT_COLS) + 1), np.float32)
    for i, (t, p) in enumerate(zip(teams, positions)):
        hist = team_hist[t][max(0, p - W):p]       # excludes the current match → no leakage
        n = len(hist)
        if n:
            X[i, W - n:, :-1] = hist
            X[i, W - n:, -1] = 1.0
    return X

y_all = df['FTR'].map({'H': 0, 'D': 1, 'A': 2}).to_numpy()
elo_h = ((df['Home_Elo'] - 1500) / 100).to_numpy(np.float32)
elo_a = ((df['Away_Elo'] - 1500) / 100).to_numpy(np.float32)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def make_dataset(W):
    Xh = build_windows(df['HomeTeam'], home_pos, W)
    Xa = build_windows(df['AwayTeam'], away_pos, W)
    data = {'n_feat': Xh.shape[2]}
    for name, ss in SPLITS.items():
        m = df['Season'].isin(ss).to_numpy()
        data[name] = {k: torch.tensor(v[m]).to(device) for k, v in
                      {'xh': Xh, 'xa': Xa, 'eh': elo_h, 'ea': elo_a, 'y': y_all}.items()}
    return data

data32 = make_dataset(32)
for s in SPLITS:
    print(s, tuple(data32[s]['xh'].shape))

train (7980, 32, 19)
val (760, 32, 19)
test (820, 32, 19)


In [ ]:
import torch.nn as nn
import torch.nn.functional as Fn

class FormElo(nn.Module):
    def __init__(self, n_feat, hidden=32, use_gru=True, use_elo=True):
        super().__init__()
        self.use_gru, self.use_elo = use_gru, use_elo
        if use_gru:
            self.gru = nn.GRU(n_feat, hidden, batch_first=True)
        in_dim = (hidden if use_gru else 0) + (2 if use_elo else 0) + 1
        self.head = nn.Linear(in_dim, 3)                  # (win, draw, loss) for that team

    def team_probs(self, x, elo_self, elo_opp, is_home):
        parts = []
        if self.use_gru:
            _, h = self.gru(x)                            # fresh hidden state per window
            parts.append(h[-1])
        if self.use_elo:
            parts += [elo_self[:, None], elo_opp[:, None]]
        parts.append(torch.full((x.shape[0], 1), is_home, device=x.device))
        return Fn.softmax(self.head(torch.cat(parts, 1)), dim=1)

    def forward(self, xh, xa, eh, ea):
        p_home = self.team_probs(xh, eh, ea, 1.0)                  # (H, D, A)
        p_away = self.team_probs(xa, ea, eh, 0.0)[:, [2, 1, 0]]    # flip to (H, D, A)
        return (p_home + p_away) / 2

In [ ]:
import time

# ================= settings =================
FAST = True                                   # False for the final run (5 seeds everywhere)
SEEDS = range(3) if FAST else range(5)
WINDOW_SEEDS = range(1) if FAST else range(3)
torch.set_num_threads(os.cpu_count())         # use all CPU cores if no GPU
print('device:', device)

# ================= metrics, training, helpers =================
def rps_each(p, y):
    """RPS per match (not averaged)."""
    o = Fn.one_hot(y, 3).float()
    return ((p.cumsum(1) - o.cumsum(1))[:, :2] ** 2).sum(1) / 2

def metrics(p, y):
    p = p.clamp_min(1e-7)
    return {'RPS': rps_each(p, y).mean().item(),
            'logloss': Fn.nll_loss(p.log(), y).item(),
            'acc': (p.argmax(1) == y).float().mean().item()}

@torch.no_grad()
def evaluate(model, d):
    model.eval()
    return metrics(model(d['xh'], d['xa'], d['eh'], d['ea']), d['y'])

def train(data, use_gru=True, use_elo=True, hidden=32, lr=3e-3, wd=1e-4,
          epochs=40, patience=5, bs=1024, seed=0):
    torch.manual_seed(seed)
    tr, va = data['train'], data['val']
    model = FormElo(data['n_feat'], hidden, use_gru, use_elo).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    best, best_state, bad = float('inf'), None, 0
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(tr['y']), device=device)
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            p = model(tr['xh'][b], tr['xa'][b], tr['eh'][b], tr['ea'][b]).clamp_min(1e-7)
            loss = Fn.nll_loss(p.log(), tr['y'][b])
            opt.zero_grad(); loss.backward(); opt.step()
        v = evaluate(model, va)['RPS']                    # early stopping on val RPS
        if v < best - 1e-5:
            best, best_state, bad = v, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return model

@torch.no_grad()
def predict(models, d):
    """Seed ensemble: average predicted probabilities."""
    for m in models:
        m.eval()
    return torch.stack([m(d['xh'], d['xa'], d['eh'], d['ea']) for m in models]).mean(0)

def compare(p_a, p_b, y, n_boot=5000):
    """Paired bootstrap of mean RPS(A) − RPS(B). Negative = A better."""
    diff = (rps_each(p_a, y) - rps_each(p_b, y)).cpu().numpy()
    boots = np.random.default_rng(0).choice(diff, (n_boot, len(diff))).mean(1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return diff.mean(), lo, hi

def baseline_rows(y_train, y):
    """Class frequencies (from train) and always-home-win, scored on y."""
    freq = torch.bincount(y_train, minlength=3).float()
    freq = (freq / freq.sum()).expand(len(y), 3)
    home = torch.zeros(len(y), 3, device=y.device); home[:, 0] = 1.0
    rows = {'Class frequencies': metrics(freq, y), 'Always home win': metrics(home, y)}
    rows['Always home win']['logloss'] = float('inf')    # P=0 on non-home results
    return rows

CONFIGS = {'Elo only': (False, True), 'GRU only': (True, False), 'GRU + Elo': (True, True)}

def fit(data, use_gru, use_elo, seeds):
    models = []
    for s in seeds:
        t0 = time.time()
        models.append(train(data, use_gru, use_elo, seed=s))
        print(f'  gru={use_gru!s:5} elo={use_elo!s:5} seed={s}  {time.time() - t0:.0f}s')
    return models

def summary(ms, d, split):
    """Ensemble metrics on `split` + train RPS (overfitting check) + seed spread."""
    return {**metrics(predict(ms, d[split]), d[split]['y']),
            'train_RPS': metrics(predict(ms, d['train']), d['train']['y'])['RPS'],
            'seed_sd': np.std([evaluate(m, d[split])['RPS'] for m in ms])}

# ================= development: VALIDATION only =================
yv = data32['val']['y']
dev_rows = baseline_rows(data32['train']['y'], yv)
trained = {}                                  # kept for the final test cell
for name, (g, e) in CONFIGS.items():
    trained[name] = fit(data32, g, e, SEEDS)
    dev_rows[name] = summary(trained[name], data32, 'val')

for W in [8, 16, 64]:
    d = make_dataset(W)
    dev_rows[f'GRU + Elo, W={W}'] = summary(fit(d, True, True, WINDOW_SEEDS), d, 'val')

display(pd.DataFrame(dev_rows).T.round(4))
for name in ['GRU only', 'GRU + Elo']:
    m, lo, hi = compare(predict(trained[name], data32['val']),
                        predict(trained['Elo only'], data32['val']), yv)
    print(f'[val] {name:10s} vs Elo only: ΔRPS = {m:+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]')

device: cuda
  gru=False elo=True  seed=0  1s
  gru=False elo=True  seed=1  1s
  gru=False elo=True  seed=2  1s
  gru=True  elo=False seed=0  1s
  gru=True  elo=False seed=1  1s
  gru=True  elo=False seed=2  1s
  gru=True  elo=True  seed=0  1s
  gru=True  elo=True  seed=1  1s
  gru=True  elo=True  seed=2  1s
  gru=True  elo=True  seed=0  1s
  gru=True  elo=True  seed=0  1s
  gru=True  elo=True  seed=0  1s


,RPS,logloss,acc,train_RPS,seed_sd
Class frequencies,0.2308,1.0527,0.4724,NaN,NaN
Always home win,0.4164,inf,0.4724,NaN,NaN
Elo only,0.1957,0.9514,0.5526,0.1991,0.0001
GRU only,0.1970,0.9582,0.5539,0.1933,0.0011
GRU + Elo,0.1954,0.9517,0.5579,0.1929,0.0010
"GRU + Elo, W=8",0.1940,0.9491,0.5711,0.1932,0.0000
"GRU + Elo, W=16",0.1942,0.9489,0.5658,0.1927,0.0000
"GRU + Elo, W=64",0.1954,0.9516,0.5632,0.1938,0.0000


[val] GRU only   vs Elo only: ΔRPS = +0.0013   95% CI [-0.0029, +0.0055]
[val] GRU + Elo  vs Elo only: ΔRPS = -0.0003   95% CI [-0.0039, +0.0032]


In [ ]:
# ================= FINAL: TEST set — run once (reuses models from the cell above) =================
yt = data32['test']['y']
test_rows = baseline_rows(data32['train']['y'], yt)
for name, ms in trained.items():
    test_rows[name] = summary(ms, data32, 'test')

display(pd.DataFrame(test_rows).T.round(4))
for name in ['GRU only', 'GRU + Elo']:
    m, lo, hi = compare(predict(trained[name], data32['test']),
                        predict(trained['Elo only'], data32['test']), yt)
    print(f'[test] {name:10s} vs Elo only: ΔRPS = {m:+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]')

,RPS,logloss,acc,train_RPS,seed_sd
Class frequencies,0.2318,1.0851,0.4146,NaN,NaN
Always home win,0.4537,inf,0.4146,NaN,NaN
Elo only,0.2062,1.0103,0.5110,0.1991,0.0003
GRU only,0.2083,1.0203,0.5012,0.1933,0.0008
GRU + Elo,0.2058,1.0114,0.5024,0.1929,0.0002


[test] GRU only   vs Elo only: ΔRPS = +0.0020   95% CI [-0.0028, +0.0068]
[test] GRU + Elo  vs Elo only: ΔRPS = -0.0005   95% CI [-0.0044, +0.0035]


In [ ]:
# Bookmaker benchmark (odds are NOT model inputs, only a reference)
odds = df[['B365H', 'B365D', 'B365A']].to_numpy(float)
p_book = (1 / odds) / (1 / odds).sum(1, keepdims=True)        # remove the margin
y_np = df['FTR'].map({'H': 0, 'D': 1, 'A': 2}).to_numpy()

rows = {}
for s in val_seasons + test_seasons:
    m = (df['Season'] == s).to_numpy() & ~np.isnan(p_book).any(1)
    rows[s] = {'matches': int(m.sum()),
               'home_win_rate': (y_np[m] == 0).mean(),
               'RPS bookmaker': metrics(torch.tensor(p_book[m]), torch.tensor(y_np[m]))['RPS']}
for split in ['val', 'test']:
    m = df['Season'].isin(SPLITS[split]).to_numpy() & ~np.isnan(p_book).any(1)
    rows[f'ALL {split}'] = {'matches': int(m.sum()),
                            'home_win_rate': (y_np[m] == 0).mean(),
                            'RPS bookmaker': metrics(torch.tensor(p_book[m]), torch.tensor(y_np[m]))['RPS']}
display(pd.DataFrame(rows).T.round(4))

,matches,home_win_rate,RPS bookmaker
2022/23,380.0,0.4842,0.1990
2023/24,380.0,0.4605,0.1839
2024/25,380.0,0.4079,0.1972
2025/26,380.0,0.4263,0.2057
2026/27,60.0,0.3833,0.2054
ALL val,760.0,0.4724,0.1914
ALL test,820.0,0.4146,0.2018
